<a href="https://colab.research.google.com/github/Rustam99-eng/Test-task-Automation-and-Analytics-Department/blob/main/%D0%A2%D0%B5%D1%81%D1%82%D0%BE%D0%B2%D0%BE%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%E2%80%94_%D0%9E%D1%82%D0%B4%D0%B5%D0%BB_%D0%B0%D0%B2%D1%82%D0%BE%D0%BC%D0%B0%D1%82%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D0%B8_%D0%B8_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D1%82%D0%B8%D0%BA%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas plotly openpyxl

In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==== Читаем файл ====
df = pd.read_excel('report_fixed.csv.xlsx', sheet_name='Лист1')
df.columns = ['client_id','project_ids','project_name','service_type','term_months',
              'flight_no','flight_start','flight_end','last_active_month',
              'status','report_generated_at']

# ==== KPI (с приведением к int!) ====
renewed = int((df['status'] == 'пролонгировано').sum())
churned = int(df['status'].isin(['непролонгировано','отвал']).sum())
other   = int((df['status'].isin(['неизвестно']) |
               df['status'].str.startswith('завершился')).sum())
base    = renewed + churned
clients = int(df['client_id'].nunique())

retention = round(renewed / base * 100) if base else 0
churn     = round(churned / base * 100) if base else 0

print(f'Уникальных клиентов: {clients}')
print(f'Пролонгация: {retention}% ({renewed}/{base})')
print(f'Отток: {churn}% ({churned}/{base})')
print(f'Прочее (разовые/неизвестно): {other}')
print(f'[debug] renewed={renewed} ({type(renewed)}), churned={churned} ({type(churned)})')

# ==== Цвета и порядок статусов ====
status_order = ['пролонгировано','непролонгировано','отвал',
                'завершился (разовые работы)','неизвестно']
color_map = {
    'пролонгировано':'#16a34a','непролонгировано':'#dc2626',
    'отвал':'#991b1b','завершился (разовые работы)':'#64748b',
    'неизвестно':'#94a3b8'
}
status_counts = df['status'].value_counts().reindex(status_order).fillna(0).astype(int)

# ==== Дашборд ====
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type':'indicator'},{'type':'indicator'}],
           [{'type':'xy'},{'type':'domain'}]],
    subplot_titles=('Клиенты','Доля пролонгации',
                    'Распределение статусов','Пролонгация vs отток'),
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

# KPI 1
fig.add_trace(go.Indicator(
    mode='number',
    value=clients,
    title='Уникальных клиентов'
), row=1, col=1)

# KPI 2
fig.add_trace(go.Indicator(
    mode='number',
    value=retention,
    number={'suffix':'%'},
    title=f'Пролонгация ({renewed}/{base})'
), row=1, col=2)

# Bar
fig.add_trace(go.Bar(
    x=status_counts.values,
    y=status_counts.index,
    orientation='h',
    marker_color=[color_map[s] for s in status_counts.index],
    showlegend=False,
    text=status_counts.values,
    textposition='outside'
), row=2, col=1)

# Pie — ВАЖНО: передаём чистые int и явно задаём domain через subplot
fig.add_trace(go.Pie(
    labels=['Пролонгировано','Ушли'],
    values=[renewed, churned],
    marker=dict(colors=['#16a34a','#dc2626']),   # ← marker=dict(colors=...), а не marker_colors
    hole=0.6,
    textinfo='label+percent',
    textposition='inside',
    showlegend=True,
    sort=False                                   # ← сохраняет порядок
), row=2, col=2)

fig.update_layout(
    height=700,
    title_text='Удержание клиентов — по исправленному отчёту',
    template='plotly_white'
)

fig.show()
fig.write_html('dashboard.html')

Уникальных клиентов: 11
Пролонгация: 20% (3/15)
Отток: 80% (12/15)
Прочее (разовые/неизвестно): 2
[debug] renewed=3 (<class 'int'>), churned=12 (<class 'int'>)
